In [ ]:
# ============================================================================
# EMOTIC Social Situation Assessment — Kaggle Benchmark Task
# ============================================================================
# Task: Given a composite image of a social scene with two people,
#        assess social dynamics: power (dominance), comfort (valence),
#        tension (arousal). IMAGE input (multimodal).
# Input: Single composite PNG (scene + person crops labeled A and B)
# Output: Per-person VAD scores (1-10), dominant person ID, tension level
# Metric: Composite score (float, 0.0 - 1.0)
# Samples: 5 multi-person scenes (deterministic, seed=42)
# ============================================================================

import kaggle_benchmarks as kbench
from kaggle_benchmarks.content_types import images
from dataclasses import dataclass
import pandas as pd
import numpy as np
from PIL import Image as PILImage
import os
import re
import json

# ============================================================================
# CONFIGURATION
# ============================================================================

DATASET_ROOT = "/kaggle/input/datasets/magdawjcicka/emotic"
NUM_SAMPLES = 250
RANDOM_SEED = 42
VAD_TOLERANCE = 2.0

VALID_TENSION_LEVELS = ["low", "medium", "high"]

TENSION_SYNONYMS = {
    "none": "low", "calm": "low", "relaxed": "low", "minimal": "low",
    "moderate": "medium", "some": "medium", "mid": "medium",
    "elevated": "high", "intense": "high", "severe": "high",
    "extreme": "high", "strong": "high",
}


# ============================================================================
# STRUCTURED OUTPUT SCHEMA
# ============================================================================

@dataclass
class SocialAssessment:
    """The LLM must return VAD scores and social dynamics assessment."""
    person_a_valence: float
    person_a_arousal: float
    person_a_dominance: float
    person_b_valence: float
    person_b_arousal: float
    person_b_dominance: float
    more_dominant_person: str
    scene_tension: str


# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def find_csv(dataset_root):
    """Locate the test annotations CSV."""
    for dirpath, _, filenames in os.walk(dataset_root):
        for fname in filenames:
            if "annot_arrs_test" in fname.lower() and fname.endswith(".csv"):
                return os.path.join(dirpath, fname)
    raise FileNotFoundError(f"annot_arrs_test.csv not found under {dataset_root}")


def find_npy_dir(dataset_root):
    """Locate the directory containing .npy image arrays."""
    for dirpath, _, filenames in os.walk(dataset_root):
        for fname in filenames:
            if fname.startswith("crop_arr_test_") and fname.endswith(".npy"):
                return dirpath
    for sub in ["img_arrs", "Emotic/img_arrs", ""]:
        candidate = os.path.join(dataset_root, sub) if sub else dataset_root
        if os.path.isdir(candidate):
            npys = [f for f in os.listdir(candidate) if f.endswith(".npy")]
            if npys:
                return candidate
    raise FileNotFoundError(f"No .npy files found under {dataset_root}")


def load_npy_as_pil(npy_path):
    """Load a .npy array and return a PIL Image."""
    arr = np.load(npy_path)
    if arr.dtype != np.uint8:
        if arr.max() <= 1.0:
            arr = (arr * 255).astype(np.uint8)
        else:
            arr = arr.astype(np.uint8)
    return PILImage.fromarray(arr)


def create_composite_png(scene_npy, crop_a_npy, crop_b_npy, out_path):
    """Combine scene + 2 crops into a single labeled PNG."""
    try:
        parts = []
        if scene_npy and os.path.exists(scene_npy):
            parts.append(("scene", load_npy_as_pil(scene_npy)))
        if crop_a_npy and os.path.exists(crop_a_npy):
            parts.append(("crop_a", load_npy_as_pil(crop_a_npy)))
        if crop_b_npy and os.path.exists(crop_b_npy):
            parts.append(("crop_b", load_npy_as_pil(crop_b_npy)))

        if not parts:
            return None

        if len(parts) == 1:
            parts[0][1].save(out_path, "PNG")
            return out_path

        W = 400
        scene = parts[0][1]
        ratio = W / scene.width
        scene = scene.resize((W, int(scene.height * ratio)), PILImage.LANCZOS)

        cw = (W - 10) // 2
        crops = []
        for label, img in parts[1:]:
            r = cw / img.width
            crops.append(img.resize((cw, min(int(img.height * r), 200)), PILImage.LANCZOS))

        crop_h = max(c.height for c in crops) if crops else 0
        total_h = 24 + scene.height + 10 + 24 + crop_h + 10
        comp = PILImage.new("RGB", (W, total_h), (255, 255, 255))

        try:
            from PIL import ImageDraw, ImageFont
            draw = ImageDraw.Draw(comp)
            try:
                font = ImageFont.truetype(
                    "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 14)
            except (IOError, OSError):
                font = ImageFont.load_default()
        except ImportError:
            draw, font = None, None

        y = 0
        if draw:
            draw.text((4, y + 4), "Full Scene", fill=(0, 0, 0), font=font)
        y += 24
        comp.paste(scene, (0, y))
        y += scene.height + 10

        if crops:
            if draw:
                draw.text((4, y + 4), "Person A", fill=(0, 0, 0), font=font)
                draw.text((cw + 14, y + 4), "Person B", fill=(0, 0, 0), font=font)
            y += 24
            x = 0
            for c in crops:
                comp.paste(c, (x, y))
                x += cw + 10

        comp.save(out_path, "PNG")
        return out_path
    except Exception as e:
        print(f"  [WARN] Composite failed: {e}")
        return None


def get_multi_person_scenes(df, n, seed):
    """Select n multi-person scenes deterministically."""
    counts = df.groupby("Filename").size()
    multi = sorted(counts[counts >= 2].index.tolist())
    if not multi:
        raise ValueError("No multi-person scenes found in dataset.")
    rng = np.random.RandomState(seed)
    return list(rng.choice(multi, size=min(n, len(multi)), replace=False))


def compute_tension(persons_df):
    """Derive tension level from aggregate VAD."""
    score = persons_df["Arousal"].mean() - (persons_df["Valence"].mean() - 5.0)
    if score < 4.5:
        return "low"
    elif score < 6.5:
        return "medium"
    return "high"


def normalize_tension(raw):
    c = re.sub(r"[^a-z ]", "", raw.strip().lower()).strip()
    if c in VALID_TENSION_LEVELS:
        return c
    if c in TENSION_SYNONYMS:
        return TENSION_SYNONYMS[c]
    for lv in VALID_TENSION_LEVELS:
        if lv in c:
            return lv
    return "INVALID"


def normalize_dominant(raw):
    c = raw.strip().upper()
    if "A" in c and "B" not in c:
        return "A"
    if "B" in c and "A" not in c:
        return "B"
    if c.startswith("A"):
        return "A"
    if c.startswith("B"):
        return "B"
    return "INVALID"


def clamp_vad(value):
    try:
        return max(1.0, min(10.0, float(value)))
    except (ValueError, TypeError):
        return 5.5


def build_prompt(person_a, person_b):
    meta_a = f"{person_a['Age']} {person_a['Gender']}"
    meta_b = f"{person_b['Age']} {person_b['Gender']}"
    bbox_a = f"[{person_a['X_min']},{person_a['Y_min']},{person_a['X_max']},{person_a['Y_max']}]"
    bbox_b = f"[{person_b['X_min']},{person_b['Y_min']},{person_b['X_max']},{person_b['Y_max']}]"

    return (
        "You are an expert social psychologist analyzing a photograph.\n\n"
        "The image shows: the full scene at top, Person A crop at bottom-left, "
        "Person B crop at bottom-right.\n\n"
        f"Person A: {meta_a}, bounding box {bbox_a}\n"
        f"Person B: {meta_b}, bounding box {bbox_b}\n\n"
        "For each person, rate on a scale of 1-10:\n"
        "- Valence: How pleasant/positive (1=very negative, 10=very positive)\n"
        "- Arousal: How activated/energized (1=very calm, 10=very agitated)\n"
        "- Dominance: How in-control (1=submissive, 10=dominant)\n\n"
        "Also determine:\n"
        "- more_dominant_person: 'A' or 'B'\n"
        "- scene_tension: 'low', 'medium', or 'high'\n\n"
        "Respond with a single JSON object containing exactly these fields:\n"
        "person_a_valence, person_a_arousal, person_a_dominance, "
        "person_b_valence, person_b_arousal, person_b_dominance, "
        "more_dominant_person, scene_tension"
    )


def parse_response(raw):
    """Parse structured or raw response into prediction dict."""
    defaults = {
        "person_a_valence": 5.5, "person_a_arousal": 5.5,
        "person_a_dominance": 5.5, "person_b_valence": 5.5,
        "person_b_arousal": 5.5, "person_b_dominance": 5.5,
        "more_dominant_person": "A", "scene_tension": "medium",
    }

    if raw is None:
        return defaults

    # Dataclass response
    if hasattr(raw, "person_a_valence"):
        return {
            "person_a_valence": clamp_vad(raw.person_a_valence),
            "person_a_arousal": clamp_vad(raw.person_a_arousal),
            "person_a_dominance": clamp_vad(raw.person_a_dominance),
            "person_b_valence": clamp_vad(raw.person_b_valence),
            "person_b_arousal": clamp_vad(raw.person_b_arousal),
            "person_b_dominance": clamp_vad(raw.person_b_dominance),
            "more_dominant_person": str(raw.more_dominant_person),
            "scene_tension": str(raw.scene_tension),
        }

    # String response — try JSON
    text = str(raw).strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    try:
        parsed = json.loads(text)
        result = {}
        for k, dv in defaults.items():
            v = parsed.get(k, dv)
            if "valence" in k or "arousal" in k or "dominance" in k:
                result[k] = clamp_vad(v)
            else:
                result[k] = str(v)
        return result
    except (json.JSONDecodeError, AttributeError):
        print(f"  [WARN] Could not parse response, using defaults.")
        return defaults


# ============================================================================
# MAIN BENCHMARK TASK
# ============================================================================

@kbench.task(
    name="emotic_social_situation_assessment",
    description=(
        "Social Situation Assessment on the EMOTIC dataset. "
        "Given a scene image with multiple people, assess power relations, "
        "comfort levels, and tension. Evaluated with deterministic hard "
        "metrics against human-annotated VAD scores."
    ),
)
def emotic_social_situation_assessment(llm) -> float:
    """
    Evaluate an LLM's ability to read social dynamics from images.
    Returns a composite score between 0.0 and 1.0.
    """

    # STEP 1: Locate files
    csv_path = find_csv(DATASET_ROOT)
    npy_dir = find_npy_dir(DATASET_ROOT)
    print(f"[INFO] CSV: {csv_path}")
    print(f"[INFO] NPY dir: {npy_dir}")

    # STEP 2: Load annotations
    df = pd.read_csv(csv_path)
    print(f"[INFO] Loaded {len(df)} rows, {df['Filename'].nunique()} unique scenes.")

    # STEP 3: Select scenes
    scene_filenames = get_multi_person_scenes(df, NUM_SAMPLES, RANDOM_SEED)
    total = len(scene_filenames)
    print(f"[INFO] Selected {total} scenes for evaluation.")

    # STEP 4: Evaluate each scene
    all_vad_passed = 0
    all_vad_total = 0
    dominant_correct = 0
    tension_correct = 0
    scene_scores = []
    results_log = []

    for i, filename in enumerate(scene_filenames):
        print(f"\n[SCENE {i+1}/{total}] {filename}")

        scene_df = df[df["Filename"] == filename].sort_values("Crop_name").reset_index(drop=True)
        person_a = scene_df.iloc[0]
        person_b = scene_df.iloc[1]

        true_dom = "A" if person_a["Dominance"] >= person_b["Dominance"] else "B"
        true_tension = compute_tension(scene_df)

        print(f"  A: V={person_a['Valence']:.1f} A={person_a['Arousal']:.1f} D={person_a['Dominance']:.1f}")
        print(f"  B: V={person_b['Valence']:.1f} A={person_b['Arousal']:.1f} D={person_b['Dominance']:.1f}")
        print(f"  GT: dominant={true_dom}, tension={true_tension}")

        # Create composite
        os.makedirs("/kaggle/working/emotic_images", exist_ok=True)
        comp_path = create_composite_png(
            os.path.join(npy_dir, person_a["Arr_name"]),
            os.path.join(npy_dir, person_a["Crop_name"]),
            os.path.join(npy_dir, person_b["Crop_name"]),
            f"/kaggle/working/emotic_images/composite_{i}.png",
        )

        prompt_text = build_prompt(person_a, person_b)

        # Call LLM — schema attempt, then fallback
        response = None
        with kbench.chats.new(f"scene_{i}"):
            try:
                if comp_path:
                    comp_img = images.from_path(comp_path)
                    response = llm.prompt(prompt_text, image=comp_img, schema=SocialAssessment)
                else:
                    response = llm.prompt(prompt_text, schema=SocialAssessment)
            except Exception as e1:
                print(f"  [WARN] Schema failed: {e1}")
                try:
                    with kbench.chats.new(f"scene_{i}_retry"):
                        if comp_path:
                            comp_img = images.from_path(comp_path)
                            response = llm.prompt(prompt_text, image=comp_img)
                        else:
                            response = llm.prompt(prompt_text)
                except Exception as e2:
                    print(f"  [ERROR] Retry failed: {e2}")
                    response = None

        pred = parse_response(response)
        pred_dom = normalize_dominant(pred["more_dominant_person"])
        pred_ten = normalize_tension(pred["scene_tension"])

        # Score VAD
        vad_pairs = [
            ("V_A", pred["person_a_valence"], person_a["Valence"]),
            ("A_A", pred["person_a_arousal"], person_a["Arousal"]),
            ("D_A", pred["person_a_dominance"], person_a["Dominance"]),
            ("V_B", pred["person_b_valence"], person_b["Valence"]),
            ("A_B", pred["person_b_arousal"], person_b["Arousal"]),
            ("D_B", pred["person_b_dominance"], person_b["Dominance"]),
        ]
        vad_pass = 0
        for name, pv, tv in vad_pairs:
            err = abs(pv - tv)
            ok = err <= VAD_TOLERANCE
            if ok:
                vad_pass += 1
            print(f"    {name}: pred={pv:.1f} true={tv:.1f} err={err:.1f} [{'PASS' if ok else 'FAIL'}]")

        all_vad_passed += vad_pass
        all_vad_total += 6

        dom_ok = pred_dom == true_dom
        ten_ok = pred_ten == true_tension
        if dom_ok:
            dominant_correct += 1
        if ten_ok:
            tension_correct += 1
        print(f"    Dominant: {'PASS' if dom_ok else 'FAIL'}  Tension: {'PASS' if ten_ok else 'FAIL'}")

        vad_score = vad_pass / 6.0
        cls_score = ((1.0 if dom_ok else 0.0) + (1.0 if ten_ok else 0.0)) / 2.0
        scene_comp = 0.60 * vad_score + 0.40 * cls_score
        scene_scores.append(scene_comp)
        print(f"    Scene score: {scene_comp:.3f}")

        results_log.append({
            "scene": i + 1, "filename": filename,
            "vad": round(vad_score, 3), "dom": dom_ok, "ten": ten_ok,
            "score": round(scene_comp, 3),
        })

    # STEP 5: Overall score
    overall_score = float(np.mean(scene_scores)) if scene_scores else 0.0

    # STEP 6: Summary
    print(f"\n{'='*60}")
    print(f"OVERALL SCORE: {overall_score:.4f}")
    print(f"VAD: {all_vad_passed}/{all_vad_total}  "
          f"Dom: {dominant_correct}/{total}  Ten: {tension_correct}/{total}")
    print(f"{'='*60}")

    # STEP 7: Assertions
    kbench.assertions.assert_true(
        0.0 <= overall_score <= 1.0,
        expectation="Composite score should be between 0 and 1."
    )
    kbench.assertions.assert_true(
        all_vad_total > 0,
        expectation="At least one VAD prediction should be evaluated."
    )
    kbench.assertions.assert_true(
        overall_score > 0.0,
        expectation="Model should achieve a non-zero score."
    )

    return overall_score


# ============================================================================
# RUN THE TASK
# ============================================================================

emotic_social_situation_assessment.run(kbench.llm)

In [ ]:
%choose emotic_social_situation_assessment